# 03. Feature Engineering V1 — 전체 성분 멀티핫

**목적**  
정규화된 전체 성분을 제품·피부타입 단위의 멀티핫 특징으로 변환합니다.

**입력**  
`data/interim/전성분_정규화_세로형.csv`

**출력**  
`data/interim/V1_전체성분_원핫인코딩.csv`

> 저장된 전처리 데이터만 사용하며 외부 요청은 발생하지 않습니다.


In [ ]:
from pathlib import Path

# Jupyter와 Colab 모두 저장소 루트에서 실행합니다.
def find_project_root(start=Path.cwd()):
    for path in [start.resolve(), *start.resolve().parents]:
        if (path / "data").is_dir() and (path / "notebooks").is_dir():
            return path
    raise FileNotFoundError("저장소를 clone한 뒤 해당 폴더 안에서 실행하세요.")

PROJECT_ROOT = find_project_root()

DATA_RAW_DIR = PROJECT_ROOT / "data" / "raw"
DATA_INTERIM_DIR = PROJECT_ROOT / "data" / "interim"
DATA_PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
REPORTS_DIR = PROJECT_ROOT / "reports"

for directory in [DATA_RAW_DIR, DATA_INTERIM_DIR, DATA_PROCESSED_DIR, REPORTS_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)


In [ ]:
import os
import pandas as pd

INPUT_PATH = DATA_INTERIM_DIR / "전성분_정규화_세로형.csv"
OUTPUT_DIR = DATA_INTERIM_DIR

print("INPUT_PATH:", INPUT_PATH)
print("OUTPUT_DIR:", OUTPUT_DIR)


### 데이터 불러오기 및 사용 컬럼만 추출

전성분_정규화_세로형.csv 전체 컬럼 중, V1(기본 전성분 원-핫 인코딩)에는
아래 4개 컬럼만 필요함:
- `product_id`: 제품 식별자 (원-핫 인코딩의 행 기준)
- `product_name_raw`, `product_name_clean`: 결과표에서 제품을 쉽게 식별하기 위한 이름 컬럼
- `canonical_name`: 식약처 표준 성분명 (원-핫 인코딩의 컬럼 기준)

canonical_name을 기준으로 쓰는 이유: ingredient_raw/ingredient_name_clean은
같은 성분인데도 표기가 여러 개로 갈리는 경우가 확인되어(오타, 공백, 이명 등),
canonical_name만 표기 혼란 없이 신뢰 가능함.


In [3]:
df = pd.read_csv(INPUT_PATH, usecols=['product_id', 'product_name_raw', 'product_name_clean', 'canonical_name'])

# 제품 이름 매핑표 (product_id 하나당 이름은 항상 하나이므로 중복 제거해서 안전하게 매핑 가능)
name_map = df[['product_id', 'product_name_raw', 'product_name_clean']].drop_duplicates(subset='product_id')

### 원-핫 인코딩

`pd.crosstab`으로 (product_id x canonical_name) 표를 만들면,
한 제품에 같은 성분이 여러 번 기재돼 있어도(중복 성분) 자동으로 하나의 셀에 잡힘.
그 다음 `(값 > 0)`으로 값을 0/1로 통일해서 "있다/없다"만 남김.


In [4]:
multi_hot = pd.crosstab(df['product_id'], df['canonical_name'])
multi_hot = (multi_hot > 0).astype(int)
multi_hot = multi_hot.reset_index()

# 제품명 컬럼 붙이기
multi_hot = name_map.merge(multi_hot, on='product_id', how='right')

print('V1 결과 shape (제품 수, 컬럼 수):', multi_hot.shape)
multi_hot.head()

V1 결과 shape (제품 수, 컬럼 수): (231, 1295)


,product_id,product_name_raw,product_name_clean,"1,2-헥산다이올","1,3-부틸렌글라이콜","2,3-부탄다이올",3-O-에틸아스코빅애씨드,4-t-부틸사이클로헥산올,4-터피네올,7-데하이드로콜레스테롤,...,흑효모발효물,흰목이버섯포자낭과추출물,흰목이버섯폴리사카라이드,흰무늬엉겅퀴추출물,흰버드나무껍질추출물,흰서양송로추출물,히비스커스꽃추출물,히스티딘,히스티딘에이치씨엘,히아신스전초추출물
0,A000000002848,[SNS 대란템]바이오더마 세비엄 포어 리파이너,바이오더마 세비엄 포어 리파이너,1,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,A000000010471,라로슈포제 에빠끌라 MAT 세보 컨트롤링 모이스춰라이저,라로슈포제 에빠끌라 MAT 세보 컨트롤링 모이스춰라이저,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,A000000117624,엠브리올리스 레 크렘 콘센트레 75ml,엠브리올리스 레 크렘 콘센트레,1,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,A000000126470,[장벽크림] 유세린 울트라 센서티브 리페어 크림 50ml,유세린 울트라 센서티브 리페어 크림,0,0,0,0,1,0,0,...,0,0,0,0,0,0,0,0,0,0
4,A000000134691,바이오더마 시카비오 포마드 100ml(리페어 리치 밤),바이오더마 시카비오 포마드 (리페어 리치 밤),0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


### 결과 저장


In [ ]:
output_path = os.path.join(OUTPUT_DIR, 'V1_전체성분_원핫인코딩.csv')
multi_hot.to_csv(output_path, index=False, encoding='utf-8-sig')
print('저장 완료:', output_path)